# Diffolio Reproduction — KF12

Training the Diffolio baseline on the Kenneth French 12 Industry Portfolios using the reproduction settings in `configs/diffolio.yaml`.

# Sync Github

In [2]:
import os
import subprocess

REPO_PATH = "/kaggle/working/diffusion-portfolio"

REPO_URL = (
    "https://github.com/"
    "Amir-Entezari/"
    "diffusion-portfolio.git"
)

if not os.path.exists(
    os.path.join(
        REPO_PATH,
        ".git",
    )
):
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            REPO_PATH,
        ],
        check=True,
    )

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "fetch",
        "origin",
        "main",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "reset",
        "--hard",
        "origin/main",
    ],
    check=True,
)

CURRENT_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        REPO_PATH,
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

os.chdir(
    REPO_PATH
)

print(
    "Synced to latest main:"
)
print(
    CURRENT_COMMIT
)

subprocess.run(
    [
        "git",
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)

Cloning into '/kaggle/working/diffusion-portfolio'...


HEAD is now at 473cbab Fix Diffolio Kaggle backup initialization
Synced to latest main:
473cbabf734e9036d5af16a0b664add8e7c74539
473cbab Fix Diffolio Kaggle backup initialization


From https://github.com/Amir-Entezari/diffusion-portfolio
 * branch            main       -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [1]:
import os
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GiB",
    )

!nvidia-smi

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB
Fri Oct  2 09:24:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
| 

In [3]:
%pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for diffusion-portfolio (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import sys

SRC_PATH = os.path.join(
    REPO_PATH,
    "src",
)

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH,
    )

import diffusion_portfolio

print(
    "Repo:",
    os.getcwd(),
)

print(
    "Source path:",
    SRC_PATH,
)

print(
    "Import successful:",
    diffusion_portfolio.__file__,
)

Repo: /kaggle/working/diffusion-portfolio
Source path: /kaggle/working/diffusion-portfolio/src
Import successful: /kaggle/working/diffusion-portfolio/src/diffusion_portfolio/__init__.py


## Artifacts
## Restore

In [ ]:
ARTIFACT_DATASET = "amirentezari/diffusion-portfolio"

!python scripts/kaggle_artifacts.py restore \
    --handle {ARTIFACT_DATASET}

## Backup

In [ ]:
EXPERIMENT = "baselines/diffolio"
RUN_DIR = "/kaggle/working/diffusion-portfolio/artifacts/diffolio"

!python scripts/kaggle_artifacts.py backup \
    --handle {ARTIFACT_DATASET} \
    --experiment {EXPERIMENT} \
    --source {RUN_DIR}

## Data

In [5]:
from pathlib import Path
import urllib.request

destination = Path(
    "data/raw/Data2025.xlsx"
)

destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)

URL = (
    "https://docs.google.com/spreadsheets/d/"
    "1g4LOaRj4TvwJr9RIaA_nwrXXWTOy46bP/"
    "export?format=xlsx"
)

urllib.request.urlretrieve(
    URL,
    destination,
)

print(
    "Downloaded to:",
    destination,
)

print(
    "Exists:",
    destination.exists(),
)

print(
    "Size:",
    destination.stat().st_size,
    "bytes",
)

Downloaded to: data/raw/Data2025.xlsx
Exists: True
Size: 474991 bytes


In [6]:
from pathlib import Path

config_path = Path(
    "configs/diffolio.yaml"
)

assert config_path.exists()

print(
    config_path.read_text()
)

seed: 42

data:
  dataset: ken_french_12
  frequency: daily

  goyal_path: "data/raw/Data2025.xlsx"

  sample_start: "1958-01-01"
  sample_end: "2023-12-31"

  train_end: "1999-12-31"
  val_end: "2004-12-31"

  lookback: 63
  horizon: 1

  return_scaling: train_zscore
  covariate_scaling: train_zscore

model:
  type: diffolio

  n_assets: 12
  n_asset_characteristics: 10
  n_systematic: 8

  hidden_dim: 128
  num_heads: 4
  mlp_dim: 512
  time_embedding_dim: 32

  prediction_type: epsilon

  diffusion_steps: 1000
  schedule: linear
  beta_start: 0.0001
  beta_end: 0.02

  lambda_corr: 0.05

training:
  batch_size: 1024

  total_steps: 100000
  warmup_steps: 1000
  max_learning_rate: 0.0001

  # Paper specifies AdamW but does not state weight decay.
  # Explicit reproduction convention: PyTorch AdamW default.
  weight_decay: 0.01

  # Not specified by the paper.
  gradient_clip_norm: null

  record_every: 100
  num_workers: 0

  checkpoint_path: "artifacts/diffolio/diffolio_100k.pt"

  

In [7]:
from pathlib import Path
import json
import shutil
import kagglehub

BACKUP_HANDLE = "amirentezari/diffolio-checkpoints"

preflight_dir = Path(
    "/kaggle/working/diffolio_backup_preflight"
)

if preflight_dir.exists():
    shutil.rmtree(
        preflight_dir
    )

preflight_dir.mkdir(
    parents=True,
    exist_ok=True,
)

preflight_file = (
    preflight_dir
    / "preflight.json"
)

with preflight_file.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        {
            "purpose": (
                "Verify durable Diffolio "
                "checkpoint uploads"
            ),
            "repository": (
                "Amir-Entezari/"
                "diffusion-portfolio"
            ),
            "commit": CURRENT_COMMIT,
        },
        handle,
        indent=2,
    )

print(
    "Uploading preflight to:",
    BACKUP_HANDLE,
)

kagglehub.dataset_upload(
    BACKUP_HANDLE,
    str(
        preflight_dir
    ),
    version_notes=(
        "Diffolio checkpoint "
        "upload preflight"
    ),
)

print(
    "Preflight upload completed."
)

Uploading preflight to: amirentezari/diffolio-checkpoints
Uploading Dataset https://api.kaggle.com/datasets/amirentezari/diffolio-checkpoints ...
Starting upload for file /kaggle/working/diffolio_backup_preflight/preflight.json

Uploading: 100%|██████████| 170/170 [00:00<00:00, 420B/s]

Upload successful: /kaggle/working/diffolio_backup_preflight/preflight.json (170B)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/amirentezari/diffolio-checkpoints
Preflight upload completed.


## Smoke test

In [8]:
!python scripts/train_diffolio.py \
    --config configs/diffolio.yaml \
    --steps 100

DIFFOLIO TRAINING
Config: configs/diffolio.yaml
Device: cuda
GPU: Tesla T4
Total steps: 100
Warmup steps: 10
Batch size: 1024
Checkpoint: artifacts/diffolio/diffolio_smoke_100.pt
NOTE: smoke-run warmup automatically adjusted to 10 steps.
PREPARING DIFFOLIO DATA
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
Raw daily returns: (16613, 12)
Train windows: 10513
Validation windows: 1256
Test windows: 4781
Training batches/epoch: 11
Training covariance: (12, 12)

Model parameters: 410,241
Trainable parameters: 410,241
Run config: artifacts/diffolio/diffolio_smoke_100_config.yaml
Preprocessing stats: artifacts/diffolio/diffolio_smoke_100_preprocessing.npz
Kaggle backup disabled for this shortened smoke run.

STARTING OPTIMIZATION
Step 000001/000100 | loss=0.963179 | ddpm=1.011710 | corr=-0.970616 | lr=1.000e-05
Step 000010/000100 

## Full training

In [ ]:
!python scripts/train_diffolio.py \
    --config configs/diffolio.yaml

DIFFOLIO TRAINING
Config: configs/diffolio.yaml
Device: cuda
GPU: Tesla T4
Total steps: 100000
Warmup steps: 1000
Batch size: 1024
Checkpoint: artifacts/diffolio/diffolio_100k.pt
PREPARING DIFFOLIO DATA
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
Raw daily returns: (16613, 12)
Train windows: 10513
Validation windows: 1256
Test windows: 4781
Training batches/epoch: 11
Training covariance: (12, 12)

Model parameters: 410,241
Trainable parameters: 410,241
Run config: artifacts/diffolio/diffolio_100k_config.yaml
Preprocessing stats: artifacts/diffolio/diffolio_100k_preprocessing.npz
Kaggle backup: amirentezari/diffolio-checkpoints
Kaggle backup every: 10,000 steps

STARTING OPTIMIZATION
Step 000001/100000 | loss=0.963179 | ddpm=1.011710 | corr=-0.970616 | lr=1.000e-07
Step 000100/100000 | loss=0.956381 | ddpm=1.004992 | corr=

Use the following cell only to resume an interrupted full run.

In [ ]:
!python scripts/train_diffolio.py \
    --config configs/diffolio.yaml \
    --resume artifacts/diffolio/diffolio_100k.pt